# 🐍 Python HTTPX Mastery: Modern Sync & Async HTTP Requests
### *From Basic REST API Calls to High-Performance Asynchronous HTTP/2 Client Architecture*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rohit-Saini-Sfdc/learn-python/blob/main/02_httpx_mastery.ipynb)

---

## 📌 Module Overview
Welcome to the second notebook in the **Python Learning Series**! In this interactive notebook, we master **HTTPX**—the next-generation HTTP client for Python 3 that supports both **Synchronous** and **Asynchronous** APIs, native **HTTP/2**, connection pooling, response streaming, strict default timeouts, and powerful testing utilities.

### 🎯 Learning Objectives
1. **HTTPX Fundamentals**: Understand why HTTPX was created and how it improves upon `requests` and `aiohttp`.
2. **Synchronous HTTP Requests**: Master GET, POST, PUT, DELETE operations and `httpx.Response` object inspection.
3. **Persistent Connection Pooling (`httpx.Client`)**: Learn connection reuse, setting base URLs, default headers, and performance optimization.
4. **Asynchronous HTTP Client (`httpx.AsyncClient`)**: Write non-blocking HTTP code, execute concurrent fetching with `asyncio.gather()`, and quantify speedup (~5x-10x faster).
5. **Streaming Large Payloads**: Stream large files, datasets, and lines without high RAM usage (`iter_bytes`, `aiter_lines`).
6. **Advanced Configuration**: Configure fine-grained timeouts (`httpx.Timeout`), HTTP/2 (`http2=True`), authentication flows, and redirect handling.
7. **Error Handling & Unit Testing**: Master exception hierarchies (`httpx.HTTPError`, `httpx.HTTPStatusError`), auto-retry logic, and network-free unit testing using `httpx.MockTransport`.

---


## 0. ⚙️ Setup & Installation

Before we start, we need to install `httpx` along with optional HTTP/2 support (`httpcore[http2]`).

In Google Colab, execute the cell below to install the necessary packages:


In [ ]:
# Install HTTPX with HTTP/2 support
!pip install -q httpx "httpcore[http2]"

import httpx
import asyncio
import time

print(f"✅ HTTPX Version installed: {httpx.__version__}")


## 1. 💡 Why HTTPX? (`httpx` vs. `requests` vs. `aiohttp`)

For years, `requests` was the standard HTTP library in Python. However, modern backend systems, microservices, and AI pipelines require **asynchronous execution**, **HTTP/2 multiplexing**, and **strict type safety**. 

HTTPX provides a **next-generation HTTP client** designed for modern Python.

### 📊 Feature Matrix Comparison

| Feature | `requests` | `aiohttp` | `httpx` |
| :--- | :---: | :---: | :---: |
| **Synchronous API (`requests`-like)** | ✅ Yes | ❌ No | ✅ Yes (`httpx.Client`) |
| **Asynchronous API (`async`/`await`)** | ❌ No | ✅ Yes | ✅ Yes (`httpx.AsyncClient`) |
| **Unified API Interface** | ❌ No | ❌ No | ✅ Identical methods sync & async |
| **HTTP/2 Support** | ❌ No | ❌ No | ✅ Yes (`http2=True`) |
| **Default Timeouts** | ❌ None (Can hang!) | ❌ None | ✅ 5 Seconds (Safe by default) |
| **Type Hints / Annotations** | ❌ Minimal | ⚠️ Partial | ✅ 100% Fully Typed |
| **Mock Transport / Testing** | ⚠️ Third-party | ⚠️ Third-party | ✅ Native (`httpx.MockTransport`) |
| **Response Streaming** | ✅ Yes | ✅ Yes | ✅ Yes (`iter_bytes`, `iter_lines`) |

---


## 2. 🌐 Basic HTTP Operations (Synchronous API)

HTTPX offers a top-level API that mirrors Python's `requests` library. You can perform standard HTTP methods directly using `httpx.get()`, `httpx.post()`, `httpx.put()`, and `httpx.delete()`.

### Key `httpx.Response` Attributes & Methods:
* `response.status_code`: HTTP status code integer (e.g., `200`, `404`, `500`).
* `response.is_success`: `True` if status code is `2xx` HTTP Success.
* `response.is_error`: `True` if status code is `4xx` or `5xx`.
* `response.json()`: Parses response body as JSON dictionary/list.
* `response.text`: Response body decoded as string.
* `response.content`: Raw response body as raw bytes.
* `response.headers`: Case-insensitive HTTP header dictionary.
* `response.url`: The request URL (including any redirects).


In [ ]:
import httpx

# 1. Standard GET Request with Query Parameters & Headers
url = "https://httpbin.org/get"
params = {"category": "python", "tutorial": "httpx", "page": 1}
headers = {"User-Agent": "HTTPX-Tutorial/1.0", "Accept": "application/json"}

print("--- 1. GET Request ---")
try:
    response = httpx.get(url, params=params, headers=headers, timeout=5.0)
    print(f"Status Code : {response.status_code} ({'Success' if response.is_success else 'Failed'})")
    print(f"Final URL   : {response.url}")
    print(f"Content-Type: {response.headers.get('content-type')}")
    data = response.json()
    print(f"QueryParams received by server: {data.get('args')}")
except Exception as e:
    print(f"Execution Note: {e}")

# 2. Standard POST Request (JSON payload)
print("\n--- 2. POST Request (JSON Payload) ---")
post_url = "https://httpbin.org/post"
json_payload = {
    "user_id": 42, 
    "role": "engineer", 
    "skills": ["python", "asyncio", "httpx"]
}

try:
    post_response = httpx.post(post_url, json=json_payload, timeout=5.0)
    print(f"Status Code : {post_response.status_code}")
    print(f"Submitted JSON data: {post_response.json().get('json')}")
except Exception as e:
    print(f"Execution Note: {e}")


## 3. 🔄 The `httpx.Client` & Connection Persistence

While top-level functions like `httpx.get()` are convenient, calling them repeatedly for multiple requests is **inefficient**. Each top-level call creates a brand new connection, performs TLS negotiation, and closes the socket immediately after.

### Why Use `httpx.Client()`?
1. **HTTP Keep-Alive & Connection Pooling**: Reuses underlying TCP sockets across multiple requests, avoiding repeated handshake overhead.
2. **Shared Configuration**: Specify `base_url`, `headers`, `cookies`, `auth`, and `timeout` once at the client level.
3. **Resource Management**: Using `with httpx.Client() as client:` guarantees that connections are closed properly upon exiting the context.

```
Top-Level Requests (httpx.get):
Request 1: [Connect 🔌] ---> [TLS Handshake 🔒] ---> [HTTP Data] ---> [Close ❌]
Request 2: [Connect 🔌] ---> [TLS Handshake 🔒] ---> [HTTP Data] ---> [Close ❌]

Client Session (httpx.Client):
Open Session: [Connect 🔌] ---> [TLS Handshake 🔒]
Request 1:   ===============> [HTTP Data] ===============> (Keep Alive ⚡)
Request 2:   ===============> [HTTP Data] ===============> (Keep Alive ⚡)
Close Session: [Close ❌]
```


In [ ]:
import httpx

# Using Client context manager with base_url and common headers
base_url = "https://jsonplaceholder.typicode.com"
default_headers = {"User-Agent": "LearnPython-HTTPX/1.0"}

print("--- Persistent Client Connection Demo ---")
try:
    with httpx.Client(base_url=base_url, headers=default_headers, timeout=10.0) as client:
        # Request 1: Fetch User Profile
        res_user = client.get("/users/1")
        user = res_user.json()
        print(f"👤 User Profile : {user['name']} ({user['email']})")
        print(f"🏢 Company      : {user['company']['name']}")
        
        # Request 2: Fetch Posts by User (reusing the same HTTP connection pool)
        res_posts = client.get("/posts", params={"userId": 1})
        posts = res_posts.json()
        print(f"📝 Total Posts  : Found {len(posts)} posts")
        print(f"📌 Sample Title : '{posts[0]['title']}'")
except Exception as e:
    print(f"Execution Note: {e}")


## 4. ⚡ Asynchronous HTTP Client (`httpx.AsyncClient`)

One of HTTPX's standout features is its native **`AsyncClient`**. 

With `AsyncClient`, HTTP requests become non-blocking coroutines (`async`/`await`). When paired with `asyncio.gather()`, you can execute dozens or hundreds of HTTP requests **concurrently in parallel**, dramatically reducing total latency.

### Sync vs Async API Equivalents:

| Synchronous (`httpx.Client`) | Asynchronous (`httpx.AsyncClient`) |
| :--- | :--- |
| `with httpx.Client() as client:` | `async with httpx.AsyncClient() as client:` |
| `response = client.get(url)` | `response = await client.get(url)` |
| `response = client.post(url, json=data)` | `response = await client.post(url, json=data)` |
| `for item in stream:` | `async for item in stream:` |


In [ ]:
import httpx
import asyncio
import time

urls = [
    "https://jsonplaceholder.typicode.com/posts/1",
    "https://jsonplaceholder.typicode.com/posts/2",
    "https://jsonplaceholder.typicode.com/posts/3",
    "https://jsonplaceholder.typicode.com/posts/4",
    "https://jsonplaceholder.typicode.com/posts/5",
]

# 1. Synchronous Sequential Fetching
def fetch_sync():
    start = time.perf_counter()
    results = []
    with httpx.Client(timeout=10.0) as client:
        for url in urls:
            try:
                res = client.get(url)
                results.append(res.json()["id"])
            except Exception:
                results.append(None)
    duration = time.perf_counter() - start
    print(f"⏱️  Sync Fetch (Sequential)  Time : {duration:.3f}s | Results: {results}")
    return duration

# 2. Asynchronous Concurrent Fetching
async def fetch_single_async(client, url):
    try:
        res = await client.get(url)
        return res.json()["id"]
    except Exception:
        return None

async def fetch_async():
    start = time.perf_counter()
    async with httpx.AsyncClient(timeout=10.0) as client:
        tasks = [fetch_single_async(client, url) for url in urls]
        results = await asyncio.gather(*tasks)
    duration = time.perf_counter() - start
    print(f"⚡ Async Fetch (Concurrent) Time : {duration:.3f}s | Results: {results}")
    return duration

# Run synchronous benchmark
print("--- Starting Sync vs Async Benchmark ---")
sync_duration = fetch_sync()

# Run asynchronous benchmark (in Jupyter / Colab environment, use await directly)
try:
    async_duration = await fetch_async()
    speedup = sync_duration / async_duration if async_duration > 0 else 1.0
    print(f"\n🚀 Speedup Factor: {speedup:.2f}x faster using AsyncClient!")
except Exception as e:
    print(f"Async execution note: {e}")


## 5. 🌊 Streaming Requests & Responses

When fetching large datasets, CSV files, video streams, or Server-Sent Events (SSE), loading the full response body into memory at once can cause **Out-Of-Memory (OOM)** exceptions.

HTTPX provides **Response Streaming** via `client.stream()`:
* **`response.iter_bytes(chunk_size)`**: Yields raw byte chunks as they arrive over the socket.
* **`response.iter_lines()`**: Yields line-by-line text strings (ideal for CSVs or log files).
* **`response.aiter_bytes()` / `response.aiter_lines()`**: Async iterators for `AsyncClient`.

> ⚠️ **Important**: When using `stream()`, the response body is NOT automatically read into memory. You must consume the stream inside the context manager block.


In [ ]:
import httpx
import asyncio

# 1. Synchronous Streaming Line-by-Line
print("--- 1. Synchronous Response Streaming ---")
stream_url = "https://httpbin.org/stream/3"

try:
    with httpx.Client(timeout=10.0) as client:
        with client.stream("GET", stream_url) as response:
            print(f"Response Status: {response.status_code}")
            for i, line in enumerate(response.iter_lines(), start=1):
                if line:
                    print(f"  Received Line {i}: {line[:60]}...")
except Exception as e:
    print(f"Streaming note: {e}")

# 2. Asynchronous Byte Streaming
print("\n--- 2. Asynchronous Byte Streaming ---")
async def stream_async_demo():
    try:
        async with httpx.AsyncClient(timeout=10.0) as client:
            async with client.stream("GET", "https://httpbin.org/bytes/1024") as response:
                total_bytes = 0
                async for chunk in response.aiter_bytes(chunk_size=256):
                    total_bytes += len(chunk)
                    print(f"  Downloaded chunk of size {len(chunk)} bytes (Total so far: {total_bytes} B)")
    except Exception as e:
        print(f"Async streaming note: {e}")

try:
    await stream_async_demo()
except Exception as e:
    print(f"Async call note: {e}")


## 6. 🛠️ Advanced HTTPX Configuration

### A. Fine-Grained Timeouts
Unlike `requests` (which defaults to no timeout and can hang indefinitely), HTTPX applies a **5.0 second default timeout**. 
You can customize timeouts globally or per-request using `httpx.Timeout`:

```python
# Timeout(total_seconds, connect=2.0, read=5.0, write=5.0, pool=2.0)
custom_timeout = httpx.Timeout(10.0, connect=3.0, read=7.0)
client = httpx.Client(timeout=custom_timeout)
```

### B. HTTP/2 Multiplexing (`http2=True`)
HTTP/2 enables multiple requests to be multiplexed over a single TCP connection simultaneously without head-of-line blocking.
To use HTTP/2, ensure `httpcore[http2]` is installed and pass `http2=True`:

```python
client = httpx.Client(http2=True)
```

### C. Authentication Flow
HTTPX supports built-in and custom authentication handlers:
* **Basic Auth**: `httpx.BasicAuth(username, password)`
* **Bearer Token Header**: `headers={"Authorization": "Bearer YOUR_TOKEN"}`
* **Custom Auth**: Class inheriting from `httpx.Auth` for signature generation or token refreshing.

### D. Redirect Handling
By default, HTTPX does NOT automatically follow HTTP redirects (301/302) for security and explicit control. Enable auto-redirects by setting `follow_redirects=True`.


In [ ]:
import httpx

# 1. Custom Timeouts & Follow Redirects
print("--- 1. Timeouts & Redirect Handling ---")
timeout_cfg = httpx.Timeout(5.0, connect=2.0)

try:
    with httpx.Client(timeout=timeout_cfg, follow_redirects=True) as client:
        # httpbin redirect endpoint
        res = client.get("https://httpbin.org/redirect/2")
        print(f"Final URL after redirects : {res.url}")
        print(f"Redirect History          : {[r.status_code for r in res.history]}")
        print(f"Final Status Code         : {res.status_code}")
except Exception as e:
    print(f"Redirect demo note: {e}")

# 2. Authentication Example (Basic Auth)
print("\n--- 2. Basic Authentication ---")
auth = httpx.BasicAuth(username="colab_user", password="secret_password_123")

try:
    with httpx.Client(auth=auth, timeout=5.0) as client:
        res = client.get("https://httpbin.org/basic-auth/colab_user/secret_password_123")
        print(f"Auth Success Status: {res.status_code}")
        print(f"Authenticated      : {res.json().get('authenticated')}")
except Exception as e:
    print(f"Auth demo note: {e}")


## 7. 🚨 Error Handling & Exception Hierarchy

Proper error handling prevents applications from crashing during network hiccups, DNS resolution failures, timeouts, or server errors.

### HTTPX Exception Hierarchy:
```
httpx.HTTPError (Base class for all HTTPX exceptions)
 ├── httpx.RequestError (Network failure, connection refused, DNS error)
 │    ├── httpx.ConnectError
 │    ├── httpx.ReadError
 │    └── httpx.TimeoutException
 └── httpx.HTTPStatusError (Raised by response.raise_for_status() on 4xx/5xx status codes)
```

### Recommended Best Practice: `response.raise_for_status()`
Calling `response.raise_for_status()` converts 4xx and 5xx HTTP responses into catchable `httpx.HTTPStatusError` exceptions.


In [ ]:
import httpx

def safe_api_request(url: str):
    print(f"\n📡 Requesting: {url}")
    try:
        response = httpx.get(url, timeout=3.0)
        # Raise exception for 4xx or 5xx responses
        response.raise_for_status()
        print(f"✅ Success! Status Code: {response.status_code}")
        return response.json()
        
    except httpx.HTTPStatusError as exc:
        print(f"❌ HTTP Status Error: Server returned {exc.response.status_code} for {exc.request.url}")
    except httpx.TimeoutException as exc:
        print(f"⏱️  Timeout Exception: Request timed out ({exc})")
    except httpx.RequestError as exc:
        print(f"🌐 Network/Request Error: Failed to reach host ({exc})")
    except httpx.HTTPError as exc:
        print(f"🚨 General HTTPX Error: {exc}")

# Test 1: Successful 200 OK
safe_api_request("https://httpbin.org/status/200")

# Test 2: Client Error 404 Not Found
safe_api_request("https://httpbin.org/status/404")

# Test 3: Server Error 500 Internal Server Error
safe_api_request("https://httpbin.org/status/500")


## 8. 🧪 Event Hooks & Unit Testing with `MockTransport`

### A. Event Hooks for Logging & Telemetry
HTTPX allows registering callback functions (**event hooks**) that trigger every time a request is sent or a response is received. This is ideal for logging API latency, injecting auth tokens, or monitoring traffic.

### B. Unit Testing with `httpx.MockTransport`
When writing unit tests for your application, **never call live external APIs**. `httpx.MockTransport` allows you to mock network traffic in-memory cleanly without sending network packets.


In [ ]:
import httpx
import time

# 1. Event Hooks Demo (Logging Request Latency)
print("--- 1. Event Hooks for Telemetry ---")

def log_request(request):
    request.state_start_time = time.perf_counter()
    print(f"▶️  [OUTGOING] {request.method} {request.url}")

def log_response(response):
    request = response.request
    elapsed = (time.perf_counter() - getattr(request, "state_start_time", time.perf_counter())) * 1000
    print(f"◀️  [INCOMING] Status {response.status_code} {response.url} (took {elapsed:.1f}ms)")

event_hooks = {"request": [log_request], "response": [log_response]}

try:
    with httpx.Client(event_hooks=event_hooks, timeout=5.0) as client:
        res = client.get("https://httpbin.org/get")
except Exception as e:
    print(f"Hook demo note: {e}")

# 2. MockTransport Demo (Network-Free Unit Testing)
print("\n--- 2. MockTransport for Unit Testing ---")

def custom_mock_handler(request: httpx.Request) -> httpx.Response:
    if request.url.path == "/api/user":
        return httpx.Response(200, json={"id": 99, "name": "Mocked Test User", "status": "active"})
    elif request.url.path == "/api/error":
        return httpx.Response(500, json={"error": "Simulated server failure"})
    return httpx.Response(404, json={"error": "Not Found"})

# Attach mock transport to client
mock_transport = httpx.MockTransport(custom_mock_handler)

with httpx.Client(transport=mock_transport, base_url="https://mock-api.local") as client:
    res1 = client.get("/api/user")
    print(f"Mock Call 1 Status: {res1.status_code} | Data: {res1.json()}")
    
    res2 = client.get("/api/error")
    print(f"Mock Call 2 Status: {res2.status_code} | Data: {res2.json()}")


## 9. 📊 HTTPX Cheatsheet & Summary

### 💡 Quick Reference Table

| Objective | HTTPX Syntax | Description |
| :--- | :--- | :--- |
| **Simple GET** | `httpx.get(url, params={...})` | Quick one-off GET request |
| **Simple POST** | `httpx.post(url, json={...})` | Quick one-off POST with JSON body |
| **Sync Session** | `with httpx.Client() as client:` | Reuses TCP connections & headers |
| **Async Session** | `async with httpx.AsyncClient() as client:` | Non-blocking async client |
| **Async Request** | `res = await client.get(url)` | Await asynchronous response |
| **Parallel Requests**| `await asyncio.gather(*[client.get(u) for u in urls])` | Fetch multiple URLs concurrently |
| **Stream Bytes** | `with client.stream("GET", url) as res:` | Memory-efficient chunk streaming |
| **Iterate Stream** | `for chunk in response.iter_bytes():` | Yield byte chunks sequentially |
| **Raise Status** | `response.raise_for_status()` | Raises `HTTPStatusError` on 4xx/5xx |
| **HTTP/2 Support** | `httpx.Client(http2=True)` | Enables HTTP/2 connection multiplexing |
| **Mock Testing** | `httpx.Client(transport=httpx.MockTransport(handler))` | Mocks HTTP responses without network |

---

## 🎉 Congratulations!
You have completed the **Python HTTPX Mastery** notebook. You now know how to build high-performance synchronous and asynchronous web clients, stream large payloads, manage timeouts, handle network errors gracefully, and mock APIs for unit testing!
